# Sparse-PA U-Net: Clean Notebook

This notebook demonstrates the  package. It is a cleaned replacement for the original Colab notebook: no  mounts, no absolute  paths, no shell cells. Paths and hyperparameters come from  (overridable via  /  env vars).

Run from the repo root, or set the notebook kernel's working directory to the repo root.

In [ ]:
import sys, os
# Make the package importable when running inside this notebook folder.
# If you installed the package (pip install -e .), the sys.path line is optional.
NOTEBOOK_DIR = os.getcwd()
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.abspath(REPO_ROOT) not in sys.path:
    sys.path.insert(0, REPO_ROOT)
from sparse_pa_unet.config import load_config, get_device, resolve_data_dirs
print('Device:', get_device())

In [ ]:
from sparse_pa_unet.models import (UNet, UNet_SE, UNet_CBAM, UNet_SimAM, UNet_SimCBAM, SparsityClassifier, MODEL_REGISTRY)
print('Models available:', list(MODEL_REGISTRY))

## Configuration

In [ ]:
config = load_config()
print('Data root:    ', config['data']['root'])
print('Checkpoint root:', config['checkpoints']['root'])
sparse_dirs, gt_dir = resolve_data_dirs(config)
print('GT dir:', gt_dir)
for sp, d in sparse_dirs.items():
    print(f'  sparse {sp}:', d)

## Dataset

In [ ]:
from sparse_pa_unet.dataset import PADataset
import os
train_ds = PADataset(os.path.join(sparse_dirs[64], 'train'), os.path.join(gt_dir, 'train'))
test_ds = PADataset(os.path.join(sparse_dirs[64], 'test'), os.path.join(gt_dir, 'test'))
print('Train samples:', len(train_ds), '| Test samples:', len(test_ds))
sparse, gt = test_ds[0]
print('Sparse shape:', tuple(sparse.shape), '| GT shape:', tuple(gt.shape), '| dtype:', sparse.dtype)

## Inference with pretrained checkpoints
Point  at your checkpoints (e.g. ) and load all models.

In [ ]:
from sparse_pa_unet.inference import load_all_models
import torch
device = get_device()
ckpt_dir = os.environ.get('CKPT_ROOT', config['checkpoints']['root'])
models = load_all_models(ckpt_dir, device)
print('Loaded models per sparsity:', list(models[64].keys()))

In [ ]:
model = models[64]['CBAM']
sparse_t, gt_t = test_ds[0]
sparse_t = sparse_t.unsqueeze(0).to(device)
with torch.no_grad():
    out = model(sparse_t).cpu().squeeze()
import numpy as np
print('Output shape:', tuple(out.shape))

## Training (single run)
Uncomment and run to train a model. Defaults mirror the notebook (30 epochs, MSE, Adam lr 1e-3).

In [ ]:
# !python -m sparse_pa_unet.train --sparsity 64 --model cbam --epochs 30
# train via subprocess for in-notebook usage:
import subprocess, sys
# subprocess.run([sys.executable, '-m', 'sparse_pa_unet.train', '--sparsity', '64', '--model', 'cbam', '--epochs', '1'])

## Evaluation

In [ ]:
from sparse_pa_unet.evaluate import evaluate_model
from sparse_pa_unet.inference import load_model
from sparse_pa_unet.models import get_model_class
from torch.utils.data import DataLoader
for sp in [32, 64, 96]:
    m = models[sp]['CBAM']
    ds = PADataset(os.path.join(sparse_dirs[sp], 'test'), os.path.join(gt_dir, 'test'))
    loader = DataLoader(ds, batch_size=8, shuffle=False)
    p, s = evaluate_model(m, loader, device)
    print(f'CBAM @ {sp} angles -> PSNR {p:.2f} dB | SSIM {s:.4f}')

## Quick sanity import check

In [ ]:
import torch
x = torch.randn(1, 1, 128, 128)
for name, cls in MODEL_REGISTRY.items():
    out = cls(in_ch=1, out_ch=1, base=32)(x)
    assert out.shape == x.shape, name
print('All model forward passes OK on 128x128.')